# From Pipeline to Agent: Voyage Cancellations Agent [Solution]

## The Next Challenge

In Session 1 you built a Travel Policy Assistant that can answer questions about Voyage's refund and cancellation rules. The support team is pleased — but they have a new request.

They don't just want the assistant to *answer* questions. They want it to *act*: look up a booking, check whether it is eligible for a refund, and process the cancellation if it is. The current pipeline cannot do any of this — it can only retrieve text and generate a response.

## Our Task

A RAG pipeline is designed to answer questions, but an agent is designed to take actions — and when a system needs to make decisions, call APIs, and complete workflows, we need an agent rather than a static retrieval pipeline. Our task today is to convert our RAG pipeline into an agentic flow.

## Smolagents vs LangGraph: Prototype and System Design

There are multiple ways to build agents in Python, and the choice often depends on the level of structure and control you need.

Smolagents is designed to make it very easy to give a model access to tools. With minimal setup, you can prototype task-oriented agents quickly. It works well when you want to experiment with tool use, explore automation ideas, or build lightweight workflows without managing the underlying execution loop yourself.

LangGraph, by contrast, makes the reasoning loop explicit. It allows you to define how tools are orchestrated, how memory is handled, and how execution progresses from step to step. This makes it better suited for more structured systems, especially when you want visibility into intermediate reasoning or need tighter control over behavior.

In practice, smolagents is often a strong choice for rapid prototyping, while LangGraph provides more flexibility for building structured, multi-step workflows. As systems grow in complexity — especially when retrieval, memory, and multi-tool coordination are involved — having explicit control over orchestration becomes increasingly important.

In [1]:
!pip install --quiet langchain-core==0.3.59 langgraph==0.4.3 langchain-openai==0.3.16 langchain-experimental==0.3.4 langgraph-supervisor==0.0.21


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.tools import tool
from langchain_core.documents import Document
from langgraph.prebuilt import create_react_agent

In [3]:
openai_key = os.environ["OPENAI"]

## Rebuild the Policy Retriever

We rebuild the vector store from Session 1 so this notebook runs standalone. The agent will call this via a `lookup_policy` tool.

In [4]:
# Import our policy text
with open("travel_policy.txt", "r") as f:
    raw_text = f.read()

# Set up the Chroma DB
headers = [("#", "Title"), ("##", "Section"), ("###", "Subsection")]

chunks = MarkdownHeaderTextSplitter(headers_to_split_on=headers).split_text(raw_text)

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small", 
                               openai_api_key=openai_key),
    collection_metadata={"hnsw:space": "cosine"}
)

# Configure the retriever
retriever = vector_db.as_retriever(
    search_type = "similarity",
    search_kwargs={"k": 3})

## Build Your First Tool

Tools are standard Python functions decorated with `@tool`. The **docstring is what the model reads** to decide when to call the tool — a clear, specific docstring leads to better decisions than a vague one.

We start with a single tool that wraps the Session 1 retriever.

**Tool 1: The Policy Lookup** This tool gives the Agent access to our RAG retriever. We write a broad docstring so the Agent knows it can use this for any policy question.

In [5]:
@tool
def lookup_policy(query: str) -> str:
    """
    Consult the official Voyage Cancellation Policy.
    Use this tool to verify refund rules or check cancellation fees.
    """

    docs = retriever.invoke(query)
    
    return "\n\n".join([d.page_content for d in docs])

In [6]:
# Define the tool set
tools = [lookup_policy]

## Wire the Agent

We now combine the language model and our tools into a single reasoning loop. The agent follows the ReAct pattern (Reason + Act): it thinks about what to do, selects a tool if needed, observes the result, and continues reasoning until it can produce a final answer. Unlike a traditional function call, this loop allows the model to dynamically decide the next step based on new information it receives from tools.

Importantly, the agent is not simply calling a tool once. It can call multiple tools in sequence, using the output of one step to inform the next. This ability to reason over intermediate results is what makes agents powerful for workflow automation.

In [7]:
llm = ChatOpenAI(model="gpt-4o-mini",
                 openai_api_key=openai_key, 
                 temperature=0)

In [8]:
# These instructions act as the strict rules of engagement for the agent.
system_prompt = """
### ROLE
You are the "Voyage Cancellations Agent." Your primary job is to process flight cancellation requests accurately and securely.

### WORKFLOW & CONSTRAINTS
1. **MANDATORY VERIFICATION:** You must NEVER cancel a ticket without first verifying the refund policy for that specific ticket class. Use the `lookup_policy` tool to check the rules.
2. **NON-REFUNDABLE TICKETS:** If the policy states a ticket is non-refundable, you must politely refuse the request and explain why. 
3. **REFUNDABLE TICKETS:** If the policy permits a refund (and any conditions like '24 hours prior' are met), you should confirm that the customer is able to cancel their ticket. 

### TONE
Professional, objective, and direct. Do not apologize for enforcing company policy.
"""

In [9]:
# Set up the agent executor
agent_executor = create_react_agent(
    llm, 
    tools, 
    prompt = system_prompt)

## Test the Policy Agent

With `verbose=True` you will see every Thought, Action, and Observation printed as the agent runs. Watch the reasoning chain — this is the ReAct loop in action.

In [10]:
query = "I want to cancel my Saver Ticket #12345."

In [11]:
for step in agent_executor.stream(
    {"messages": [("user", query)]},
    stream_mode="updates",
):
    print("\n--- STEP ---")
    print(step)


--- STEP ---
{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_JYJTfnfhSuEufANtkx05xk7g', 'function': {'arguments': '{"query":"Saver Ticket"}', 'name': 'lookup_policy'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 236, 'total_tokens': 251, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_3620f71984', 'id': 'chatcmpl-Df56YGB2QlYndSlDSYnh0mfi9SglR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--3bdee9d4-5387-48f3-8b91-c62a07739155-0', tool_calls=[{'name': 'lookup_policy', 'args': {'query': 'Saver Ticket'}, 'id': 'call_JYJTfnfhSuEufANtkx05xk7g', 'type': 'tool_call'}], usage_metadata={'input_tokens': 236, 'output_tok

In [15]:
query = "I want to cancel my Main Cabin Ticket #9999. It is for next week."

In [16]:
for step in agent_executor.stream(
    {"messages": [("user", query)]},
    stream_mode="updates",
):
    print("\n--- STEP ---")
    print(step)


--- STEP ---
{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_67DyeOe2Fo5CvDzH3EgNjMkd', 'function': {'arguments': '{"query":"Main Cabin Ticket"}', 'name': 'lookup_policy'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 243, 'total_tokens': 259, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_3620f71984', 'id': 'chatcmpl-Df5NoqQLYpB4GxrqgU9reKICMtJ6H', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--84de6017-4427-465a-82c4-270c4b89b303-0', tool_calls=[{'name': 'lookup_policy', 'args': {'query': 'Main Cabin Ticket'}, 'id': 'call_67DyeOe2Fo5CvDzH3EgNjMkd', 'type': 'tool_call'}], usage_metadata={'input_tokens': 243, '

## Expanding the Toolkit

Policy lookup alone is not enough. To process a cancellation the agent also needs to:
1. **Look up the booking** to find out the ticket type.
2. **Cancel the ticket** once the policy confirms it is eligible.

We add two more tools. Notice how the docstrings guide the agent's decision on *when* to use each one.

In [17]:
@tool
def cancel_ticket(ticket_id: str) -> str:
    """
    Cancels a flight booking immediately.
    WARNING: You must use the 'lookup_policy' tool to verify the ticket is refundable BEFORE calling this tool. Do not cancel non-refundable tickets.
    """
    # In a real application, this would send an API request to the booking system.
    return f"SUCCESS: Ticket #{ticket_id} has been cancelled."

In [18]:
tools = [lookup_policy, cancel_ticket]

## Wire the Full Agent

We now rebuild the agent with the new workflow.

In [19]:
system_prompt = """
### ROLE
You are the "Voyage Cancellations Agent." Your primary job is to process flight cancellation requests accurately and securely.

### WORKFLOW & CONSTRAINTS
1. **MANDATORY VERIFICATION:** You must NEVER cancel a ticket without first verifying the refund policy for that specific ticket class. Use the `lookup_policy` tool to check the rules.
2. **NON-REFUNDABLE TICKETS:** If the policy states a ticket is non-refundable, you must politely refuse the request and explain why. Do NOT call the cancellation tool.
3. **REFUNDABLE TICKETS:** If the policy permits a refund (and any conditions like '24 hours prior' are met), you should proceed to call the `cancel_ticket` tool.

### TONE
Professional, objective, and direct. Do not apologize for enforcing company policy.
"""

In [20]:
agent_executor = create_react_agent(
    llm, 
    tools, 
    prompt = system_prompt)

In [21]:
query = "I want to cancel my Standard Ticket #9999. It is for next week."

In [22]:
for step in agent_executor.stream(
    {"messages": [("user", query)]},
    stream_mode="updates",
):
    print("\n--- STEP ---")
    print(step)


--- STEP ---
{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_nVB8kjiplu03XVcwpERAYFAL', 'function': {'arguments': '{"query":"Standard Ticket"}', 'name': 'lookup_policy'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 297, 'total_tokens': 312, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a1b343a790', 'id': 'chatcmpl-Df5O6ufudkMw53a1jd4F6XLicnWi0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--e92999cd-5441-4359-a7ab-516234b0e796-0', tool_calls=[{'name': 'lookup_policy', 'args': {'query': 'Standard Ticket'}, 'id': 'call_nVB8kjiplu03XVcwpERAYFAL', 'type': 'tool_call'}], usage_metadata={'input_tokens': 297, 'outp

So far, we have tested the agent on valid cancellation requests. But what happens if a user asks something unrelated to Voyage policies?

## Persistent Memory

Up to this point, each request has been treated as an independent interaction. The agent has no awareness of previous messages. In real applications, however, users ask follow-up questions that rely on earlier context. To support this, we attach a memory layer to the agent.

In [23]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

In [24]:
memory = MemorySaver()

In [25]:
agent = create_react_agent(
    llm,
    tools=[lookup_policy],
    checkpointer=memory
)

In LangGraph, memory is organized by `thread_id`. Each thread represents a separate conversation. Messages within the same thread share context. Messages in different threads remain completely isolated.

In [26]:
config = {"configurable": {"thread_id": "user_1"}}

In [27]:
response = agent.invoke(
    {"messages": [("user", "I want to cancel my Saver Ticket #12345.")]},
    config=config,
)

In [28]:
response["messages"][-1].content

'The cancellation policy for your Saver Ticket is as follows:\n\n- **Refund Status:** Non-refundable.\n- **Changes:** Not permitted.\n- **Exceptions:** If the airline cancels the flight, a full refund will be issued.\n- **Note:** Saver fares do not include checked baggage.\n\nHowever, if you cancel your ticket:\n- **More than 48 hours before departure:** 100% refundable to the original payment method.\n- **Within 48 hours of departure:** A 50% cancellation fee applies, and the remaining 50% will be issued as Voyage Credits.\n- **Within 4 hours of departure:** Non-refundable.\n\nSince Saver Tickets are generally non-refundable, you may want to consider the timing of your cancellation to see if you can receive any credits or refunds based on the above conditions. Would you like to proceed with the cancellation?'

In [29]:
config2 = {"configurable": {"thread_id": "user_2"}}

response = agent.invoke(
    {"messages": [("user", "I want to cancel my Business class ticket for a flight tomorrow.")]},
    config=config2,
)

response["messages"][-1].content

'Since your flight is tomorrow, the cancellation policy for your Business class ticket is as follows:\n\n- **Within 48 hours of departure:** A 50% cancellation fee applies. The remaining 50% will be issued as Voyage Credits.\n- **Within 4 hours of departure:** The ticket is non-refundable.\n\nIf you decide to cancel, you will receive 50% of the ticket price back as Voyage Credits. Would you like to proceed with the cancellation?'

In [30]:
response = agent.invoke(
    {"messages": [("user", "So I can't get a refund?")]},
    config=config,
)

In [31]:
response["messages"][-1].content

'You can get a refund under certain conditions:\n\n1. **More than 48 hours before departure:** You can receive a 100% refund to your original payment method.\n2. **Within 48 hours of departure:** You will incur a 50% cancellation fee, and the remaining 50% will be issued as Voyage Credits.\n3. **Within 4 hours of departure:** The ticket is non-refundable.\n\nIf you are within 48 hours of your flight, you will only receive 50% of the ticket price back as Voyage Credits. If you are more than 48 hours away from your flight, you can get a full refund. \n\nPlease let me know if you would like to proceed with the cancellation or if you have any other questions!'

In [32]:
response = agent.invoke(
    {"messages": [("user", "Yes, I would like to cancel anyway")]},
    config=config2,
)

response["messages"][-1].content

'Please provide me with your booking reference or any other details related to your ticket so I can assist you with the cancellation process.'